NOTEBOOK 3: Feature Engineering
Run AFTER Notebook 2 is complete.
 
Inputs:
    - capitol_trades_clean.csv
    - all_stocks_clean.csv
 
Outputs:
    - analytical_dataset.csv   (one row per trade, all features + Y label)
    - dropped_trades_log.csv   (trades dropped and why — for your report)
 
This is the most important notebook. Every feature will be use in subsequent notebooks.

# CELL 1 — Imports & Load Data

In [2]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

TRADES_PATH    = Path("data") / "processed" / "cleaned" / "capitol_trades_clean.csv"
STOCKS_PATH    = Path("data") / "processed" / "cleaned" / "all_stocks_clean.csv"
OUTPUT_MAIN    = Path("data") / "output" / "analytical_dataset.csv"
OUTPUT_DROPPED = Path("data") / "output" / "dropped_trades_log.csv"

LOOKBACK_DAYS     = 180   # days for historical baseline μ and σ
MIN_LOOKBACK_OBS  = 120   # minimum trading days needed in lookback window
CONTROL_OFFSET    = 90    # days before T0 where control window ends

OUTPUT_MAIN.parent.mkdir(parents=True, exist_ok=True)

trades = pd.read_csv(TRADES_PATH, parse_dates=['trade_date', 'disclosure_date'])
stocks = pd.read_csv(STOCKS_PATH, parse_dates=['date'])

stocks = stocks.sort_values(['ticker', 'date']).reset_index(drop=True)

print("=" * 60)
print("DATA LOADED")
print(f"  Trades : {len(trades):,} rows")
print(f"  Stocks : {len(stocks):,} rows | {stocks['ticker'].nunique():,} tickers")
print(f"  Trade date range : {trades['trade_date'].min().date()} → "
      f"{trades['trade_date'].max().date()}")
print(f"  Stock date range : {stocks['date'].min().date()} → "
      f"{stocks['date'].max().date()}")
print("=" * 60)

DATA LOADED
  Trades : 26,566 rows
  Stocks : 1,254,044 rows | 1,566 tickers
  Trade date range : 2023-04-17 → 2026-03-31
  Stock date range : 2023-01-02 → 2026-12-03


# CELL 2 — Build Ticker Price Lookup Dictionary

In [3]:
print("Building ticker lookup dictionary...")
 
ticker_data = {}
for ticker, group in stocks.groupby('ticker'):
    df = group[['date', 'close']].copy()
    df = df.dropna(subset=['close'])
    df = df.sort_values('date').reset_index(drop=True)
    # Compute log returns once per ticker upfront
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    ticker_data[ticker] = df
 
print(f"Lookup dictionary built for {len(ticker_data):,} tickers")
 
# Also build SPY lookup separately for Y label calculation
if 'SPY' not in ticker_data:
    raise ValueError("SPY not found in stocks data — cannot compute Y label. "
                     "Check all_stocks_clean.csv")
 
spy_df = ticker_data['SPY'].copy()
print(f"SPY data: {len(spy_df):,} rows | "
      f"{spy_df['date'].min().date()} → {spy_df['date'].max().date()}")

Building ticker lookup dictionary...
Lookup dictionary built for 1,566 tickers
SPY data: 813 rows | 2023-01-02 → 2026-12-03


# CELL 3 — Core Feature Engineering Functions

In [4]:
def get_price_window(ticker_df, start_date, end_date):
    """
    Return rows from ticker_df between start_date and end_date (inclusive).
    Returns empty DataFrame if no data found.
    """
    mask = (ticker_df['date'] >= start_date) & (ticker_df['date'] <= end_date)
    return ticker_df[mask].copy()
 
 
def compute_baseline(ticker_df, t0, lookback_days=180):
    """
    Compute historical mean (mu) and std dev (sigma) of log returns
    over the lookback window [T0 - lookback_days, T0 - 1].
 
    Returns: (mu, sigma, n_obs) or (None, None, 0) if insufficient data.
    """
    end_date   = t0 - pd.Timedelta(days=1)
    start_date = t0 - pd.Timedelta(days=lookback_days)
 
    window = get_price_window(ticker_df, start_date, end_date)
    returns = window['log_return'].dropna()
 
    if len(returns) < MIN_LOOKBACK_OBS:
        return None, None, len(returns)
 
    return returns.mean(), returns.std(), len(returns)
 
 
def compute_abnormal_returns(ticker_df, start_date, end_date, mu, sigma):
    """
    Compute AbnRet_t for every trading day in [start_date, end_date].
    AbnRet_t = (log_return_t - mu) / sigma
 
    Returns DataFrame with columns: date, log_return, abn_ret
    Returns empty DataFrame if no data in window.
    """
    window = get_price_window(ticker_df, start_date, end_date)
    window = window.dropna(subset=['log_return'])
 
    if len(window) == 0 or sigma == 0 or sigma is None:
        return pd.DataFrame()
 
    window['abn_ret'] = (window['log_return'] - mu) / sigma
    return window[['date', 'log_return', 'abn_ret']]
 
 
def compute_stock_return(ticker_df, t0, n_days=60):
    """
    Compute cumulative log return from T0 over the next n_days trading days.
    Uses actual trading days (not calendar days).
 
    Returns: float or None if insufficient data.
    """
    # Get rows after T0
    future = ticker_df[ticker_df['date'] > t0].head(n_days)
 
    if len(future) < n_days * 0.8:   # need at least 80% of expected days
        return None
 
    start_price = ticker_df[ticker_df['date'] <= t0]['close'].iloc[-1] \
                  if len(ticker_df[ticker_df['date'] <= t0]) > 0 else None
    end_price   = future['close'].iloc[-1] if len(future) > 0 else None
 
    if start_price is None or end_price is None or start_price == 0:
        return None
 
    return np.log(end_price / start_price)
 
 
print("Feature engineering functions defined.")
print(f"  Lookback window  : {LOOKBACK_DAYS} calendar days")
print(f"  Min observations : {MIN_LOOKBACK_OBS} trading days")
print(f"  Control offset   : {CONTROL_OFFSET} days before T0")

Feature engineering functions defined.
  Lookback window  : 180 calendar days
  Min observations : 120 trading days
  Control offset   : 90 days before T0


# CELL 4 — Main Feature Engineering Loop

In [5]:
print("=" * 60)
print("STARTING FEATURE ENGINEERING LOOP")
print(f"Processing {len(trades):,} trades...")
print("=" * 60)
 
results   = []
dropped   = []
 
for i, row in trades.iterrows():
 
    ticker     = row['ticker']
    t0         = row['trade_date']
    t_disc     = row['disclosure_date']
    blind_days = row['filed_after_days']
 
    #Progress report every 1000 rows
    if i % 1000 == 0:
        print(f"  [{i:>6,} / {len(trades):,}]  Results so far: {len(results):,}")
 
    #ticker must exist in our price data
    if ticker not in ticker_data:
        dropped.append({**row.to_dict(),
                        'drop_reason': 'ticker not in stock data'})
        continue
 
    tdf = ticker_data[ticker]
 
    #Compute historical baseline
    mu, sigma, n_obs = compute_baseline(tdf, t0, LOOKBACK_DAYS)
 
    if mu is None:
        dropped.append({**row.to_dict(),
                        'drop_reason': f'insufficient lookback data ({n_obs} obs < {MIN_LOOKBACK_OBS})'})
        continue
 
    if sigma == 0:
        dropped.append({**row.to_dict(),
                        'drop_reason': 'sigma = 0 (no price movement in lookback window)'})
        continue
 
    #Compute blind spot AbnRet
    blind_spot_df = compute_abnormal_returns(tdf, t0, t_disc, mu, sigma)
 
    if len(blind_spot_df) == 0:
        dropped.append({**row.to_dict(),
                        'drop_reason': 'no price data in blind spot window'})
        continue
 
    abn_ret_max   = blind_spot_df['abn_ret'].max()
    abn_ret_mean  = blind_spot_df['abn_ret'].mean()
    abn_ret_min   = blind_spot_df['abn_ret'].min()
    n_blind_days  = len(blind_spot_df)
 
    # Anomaly flags
    anomaly_L1 = int((blind_spot_df['abn_ret'] > 1.96).any())   # 95th pct
    anomaly_L2 = int((blind_spot_df['abn_ret'] > 2.58).any())   # 99th pct
    n_anomaly_days_L1 = (blind_spot_df['abn_ret'] > 1.96).sum()
 
    # Compute control window AbnRet
    # Control window: same length as blind spot, ending 90 days before T0
    ctrl_end   = t0 - pd.Timedelta(days=CONTROL_OFFSET)
    ctrl_start = ctrl_end - pd.Timedelta(days=int(blind_days))
 
    ctrl_df = compute_abnormal_returns(tdf, ctrl_start, ctrl_end, mu, sigma)
 
    ctrl_abn_ret_mean = ctrl_df['abn_ret'].mean() if len(ctrl_df) > 0 else np.nan
    ctrl_abn_ret_max  = ctrl_df['abn_ret'].max()  if len(ctrl_df) > 0 else np.nan
    n_ctrl_days       = len(ctrl_df)
 
    #Compute sigma_30 (pre-trade 30-day volatility)
    sigma30_start = t0 - pd.Timedelta(days=30)
    sigma30_window = get_price_window(tdf, sigma30_start, t0)
    sigma_30 = sigma30_window['log_return'].dropna().std() \
               if len(sigma30_window) >= 10 else np.nan
 
    #(beat SPY over 60 days)
    stock_return_60d = compute_stock_return(tdf, t0, n_days=60)
    spy_return_60d   = compute_stock_return(spy_df, t0, n_days=60)
 
    if stock_return_60d is not None and spy_return_60d is not None:
        y_label = int(stock_return_60d > spy_return_60d)
        alpha_60d = stock_return_60d - spy_return_60d
    else:
        y_label   = np.nan   # will be dropped in Notebook 6 only
        alpha_60d = np.nan
 
    #Assemble result row
    results.append({
        # Identity columns
        'politician':           row['politician'],
        'party':                row['party'],
        'chamber':              row['chamber'],
        'state':                row['state'],
        'company':              row['company'],
        'ticker':               ticker,
        'trade_date':           t0,
        'disclosure_date':      t_disc,
        'owner':                row['owner'],
        'trade_type':           row['trade_type'],
 
        # Encoded features (for ML models)
        'trade_type_encoded':   row['trade_type_encoded'],
        'size_bracket':         row['size_bracket'],
        'size_bracket_ordinal': row['size_bracket_ordinal'],
 
        # Baseline statistics
        'mu_historical':        round(mu, 8),
        'sigma_historical':     round(sigma, 8),
        'n_lookback_obs':       n_obs,
 
        # Blind spot features
        'filed_after_days':     blind_days,
        'n_blind_spot_days':    n_blind_days,
        'abn_ret_mean':         round(abn_ret_mean, 6),
        'abn_ret_max':          round(abn_ret_max, 6),
        'abn_ret_min':          round(abn_ret_min, 6),
        'anomaly_flag_L1':      anomaly_L1,
        'anomaly_flag_L2':      anomaly_L2,
        'n_anomaly_days_L1':    n_anomaly_days_L1,
 
        # Control window features
        'ctrl_abn_ret_mean':    round(ctrl_abn_ret_mean, 6) if not np.isnan(ctrl_abn_ret_mean) else np.nan,
        'ctrl_abn_ret_max':     round(ctrl_abn_ret_max, 6)  if not np.isnan(ctrl_abn_ret_max)  else np.nan,
        'n_ctrl_days':          n_ctrl_days,
 
        # Additional features for ML
        'sigma_30':             round(sigma_30, 8) if not np.isnan(sigma_30) else np.nan,
 
        # Y label (supervised learning target)
        'stock_return_60d':     round(stock_return_60d, 6) if stock_return_60d is not None else np.nan,
        'spy_return_60d':       round(spy_return_60d, 6)   if spy_return_60d   is not None else np.nan,
        'alpha_60d':            round(alpha_60d, 6)         if not np.isnan(alpha_60d)       else np.nan,
        'Y':                    y_label,
    })
 
print(f"\nLoop complete.")
print(f"  Rows produced : {len(results):,}")
print(f"  Rows dropped  : {len(dropped):,}")

STARTING FEATURE ENGINEERING LOOP
Processing 26,566 trades...
  [     0 / 26,566]  Results so far: 0
  [ 1,000 / 26,566]  Results so far: 981
  [ 2,000 / 26,566]  Results so far: 1,961
  [ 3,000 / 26,566]  Results so far: 2,935
  [ 4,000 / 26,566]  Results so far: 3,919
  [ 5,000 / 26,566]  Results so far: 4,895
  [ 6,000 / 26,566]  Results so far: 5,861
  [ 7,000 / 26,566]  Results so far: 6,835
  [ 8,000 / 26,566]  Results so far: 7,803
  [ 9,000 / 26,566]  Results so far: 8,785
  [10,000 / 26,566]  Results so far: 9,751
  [11,000 / 26,566]  Results so far: 10,731
  [12,000 / 26,566]  Results so far: 11,711
  [13,000 / 26,566]  Results so far: 12,679
  [14,000 / 26,566]  Results so far: 13,649
  [15,000 / 26,566]  Results so far: 14,615
  [16,000 / 26,566]  Results so far: 15,555
  [17,000 / 26,566]  Results so far: 16,505
  [18,000 / 26,566]  Results so far: 17,424
  [19,000 / 26,566]  Results so far: 18,385
  [20,000 / 26,566]  Results so far: 19,334
  [21,000 / 26,566]  Results so

# CELL 5 — Convert to DataFrame & Validate

In [6]:
analytical = pd.DataFrame(results)
dropped_df = pd.DataFrame(dropped)
 
print("=" * 60)
print("ANALYTICAL DATASET OVERVIEW")
print("=" * 60)
print(f"Shape: {analytical.shape}")
print(f"\nColumn list:")
for col in analytical.columns:
    null_count = analytical[col].isna().sum()
    print(f"  {col:<35} nulls: {null_count:,}")

ANALYTICAL DATASET OVERVIEW
Shape: (24130, 32)

Column list:
  politician                          nulls: 0
  party                               nulls: 38
  chamber                             nulls: 38
  state                               nulls: 38
  company                             nulls: 0
  ticker                              nulls: 0
  trade_date                          nulls: 0
  disclosure_date                     nulls: 0
  owner                               nulls: 0
  trade_type                          nulls: 0
  trade_type_encoded                  nulls: 0
  size_bracket                        nulls: 0
  size_bracket_ordinal                nulls: 0
  mu_historical                       nulls: 0
  sigma_historical                    nulls: 0
  n_lookback_obs                      nulls: 0
  filed_after_days                    nulls: 0
  n_blind_spot_days                   nulls: 0
  abn_ret_mean                        nulls: 0
  abn_ret_max                         nulls

# CELL 6 — Drop Reason Summary

In [7]:
print("\n── WHY TRADES WERE DROPPED")
if len(dropped_df) > 0:
    reason_counts = dropped_df['drop_reason'].value_counts()
    for reason, count in reason_counts.items():
        pct = count / len(trades) * 100
        print(f"  {reason:<55} {count:>5,}  ({pct:.1f}%)")
else:
    print("  No trades dropped.")
 
print(f"\nTotal dropped : {len(dropped_df):,} / {len(trades):,} "
      f"({len(dropped_df)/len(trades)*100:.1f}%)")
print(f"Total retained: {len(analytical):,} / {len(trades):,} "
      f"({len(analytical)/len(trades)*100:.1f}%)")
 


── WHY TRADES WERE DROPPED
  ticker not in stock data                                  898  (3.4%)
  insufficient lookback data (96 obs < 120)                 278  (1.0%)
  insufficient lookback data (118 obs < 120)                113  (0.4%)
  insufficient lookback data (94 obs < 120)                 106  (0.4%)
  insufficient lookback data (87 obs < 120)                  90  (0.3%)
  insufficient lookback data (79 obs < 120)                  75  (0.3%)
  insufficient lookback data (98 obs < 120)                  74  (0.3%)
  insufficient lookback data (89 obs < 120)                  66  (0.2%)
  insufficient lookback data (76 obs < 120)                  55  (0.2%)
  insufficient lookback data (83 obs < 120)                  38  (0.1%)
  insufficient lookback data (77 obs < 120)                  30  (0.1%)
  insufficient lookback data (116 obs < 120)                 29  (0.1%)
  insufficient lookback data (115 obs < 120)                 24  (0.1%)
  insufficient lookback data (104 ob

# CELL 7 — Compute Rolling Politician AbnRet (anti-leakage feature for Random Forest)

In [8]:
# For each trade, compute the politician's average AbnRet
# from all their PRIOR trades only (no look-ahead).
# This is the key anti-leakage step for the ML model.
 
print("\nComputing rolling politician AbnRet (anti-leakage)...")
 
analytical = analytical.sort_values(['politician', 'trade_date']).reset_index(drop=True)
 
rolling_abn = []
for politician, group in analytical.groupby('politician'):
    group = group.sort_values('trade_date')
    # For each row, compute mean of all PREVIOUS rows for that politician
    expanding_mean = group['abn_ret_mean'].expanding().mean().shift(1)
    rolling_abn.append(expanding_mean)
 
analytical['politician_rolling_abn_ret'] = pd.concat(rolling_abn)
 
# First trade per politician has no history — fill with 0 (neutral)
analytical['politician_rolling_abn_ret'] = \
    analytical['politician_rolling_abn_ret'].fillna(0)
 
print(f"Rolling AbnRet computed. NaN count: "
      f"{analytical['politician_rolling_abn_ret'].isna().sum()}")


Computing rolling politician AbnRet (anti-leakage)...
Rolling AbnRet computed. NaN count: 0


# CELL 8 — Key Statistics Summary

In [10]:
print("=" * 60)
print("KEY FEATURE STATISTICS")
print("=" * 60)
 
print(f"\n── Blind Spot AbnRet")
print(analytical['abn_ret_mean'].describe().round(4).to_string())
 
print(f"\n── Control Window AbnRet")
print(analytical['ctrl_abn_ret_mean'].describe().round(4).to_string())
 
print(f"\n── Anomaly Flag Summary ")
l1_pct = analytical['anomaly_flag_L1'].mean() * 100
l2_pct = analytical['anomaly_flag_L2'].mean() * 100
print(f"  Trades with Level 1 anomaly (>1.96) : "
      f"{analytical['anomaly_flag_L1'].sum():,}  ({l1_pct:.1f}%)")
print(f"  Trades with Level 2 anomaly (>2.58) : "
      f"{analytical['anomaly_flag_L2'].sum():,}  ({l2_pct:.1f}%)")
 
print(f"\n── Y Label Distribution")
y_valid = analytical['Y'].dropna()
print(f"  Trades with Y label computed : {len(y_valid):,}")
print(f"  Y = 1 (beat SPY)             : {(y_valid == 1).sum():,}  "
      f"({(y_valid == 1).mean()*100:.1f}%)")
print(f"  Y = 0 (did not beat SPY)     : {(y_valid == 0).sum():,}  "
      f"({(y_valid == 0).mean()*100:.1f}%)")
print(f"  Trades missing Y label       : {analytical['Y'].isna().sum():,}")
print(f"  (Missing Y = trades too recent for 60-day forward window)")
 
print(f"\n── Blind Spot vs Control Comparison")
valid = analytical.dropna(subset=['abn_ret_mean', 'ctrl_abn_ret_mean'])
print(f"  Mean blind spot AbnRet  : {valid['abn_ret_mean'].mean():>8.4f}")
print(f"  Mean control AbnRet     : {valid['ctrl_abn_ret_mean'].mean():>8.4f}")
diff = valid['abn_ret_mean'].mean() - valid['ctrl_abn_ret_mean'].mean()
direction = "HIGHER" if diff > 0 else "LOWER"
print(f"  Difference              : {diff:>8.4f}  ← blind spot is {direction}")

 

KEY FEATURE STATISTICS

── Blind Spot AbnRet
count    24130.0000
mean         0.0109
std          0.2988
min         -4.3540
25%         -0.1498
50%          0.0151
75%          0.1751
max          4.1042

── Control Window AbnRet
count    24130.0000
mean         0.0069
std          0.2115
min         -1.7282
25%         -0.1204
50%          0.0109
75%          0.1354
max          1.1405

── Anomaly Flag Summary 
  Trades with Level 1 anomaly (>1.96) : 10,306  (42.7%)
  Trades with Level 2 anomaly (>2.58) : 6,515  (27.0%)

── Y Label Distribution
  Trades with Y label computed : 22,266
  Y = 1 (beat SPY)             : 10,464  (47.0%)
  Y = 0 (did not beat SPY)     : 11,802  (53.0%)
  Trades missing Y label       : 1,864
  (Missing Y = trades too recent for 60-day forward window)

── Blind Spot vs Control Comparison
  Mean blind spot AbnRet  :   0.0109
  Mean control AbnRet     :   0.0069
  Difference              :   0.0040  ← blind spot is HIGHER


# CELL 9 — Save Outputs

In [14]:
analytical.to_csv(OUTPUT_MAIN, index=False)
if len(dropped_df) > 0:
    dropped_df.to_csv(OUTPUT_DROPPED, index=False)

print("=" * 60)
print("SAVED OUTPUTS")
print("=" * 60)
print(f"  {str(OUTPUT_MAIN):<40} {len(analytical):,} rows")
if len(dropped_df) > 0:
    print(f"  {str(OUTPUT_DROPPED):<40} {len(dropped_df):,} rows")

print(f"\nColumns in analytical_dataset.csv:")
print([c for c in analytical.columns])

print("\nNotebook 3 complete. Proceed to Notebook 4.")
print("\nMaybe use for report:")
print(f"  Final analytical dataset rows : {len(analytical):,}")
print(f"  Trades retained               : {len(analytical)/len(trades)*100:.1f}%")
print(f"  Trades with Y label           : {analytical['Y'].notna().sum():,}")
print(f"  Level 1 anomaly rate          : {l1_pct:.1f}%")
print(f"  Level 2 anomaly rate          : {l2_pct:.1f}%")


SAVED OUTPUTS
  data/output/analytical_dataset.csv       24,130 rows
  data/output/dropped_trades_log.csv       2,436 rows

Columns in analytical_dataset.csv:
['politician', 'party', 'chamber', 'state', 'company', 'ticker', 'trade_date', 'disclosure_date', 'owner', 'trade_type', 'trade_type_encoded', 'size_bracket', 'size_bracket_ordinal', 'mu_historical', 'sigma_historical', 'n_lookback_obs', 'filed_after_days', 'n_blind_spot_days', 'abn_ret_mean', 'abn_ret_max', 'abn_ret_min', 'anomaly_flag_L1', 'anomaly_flag_L2', 'n_anomaly_days_L1', 'ctrl_abn_ret_mean', 'ctrl_abn_ret_max', 'n_ctrl_days', 'sigma_30', 'stock_return_60d', 'spy_return_60d', 'alpha_60d', 'Y', 'politician_rolling_abn_ret']

Notebook 3 complete. Proceed to Notebook 4.

Maybe use for report:
  Final analytical dataset rows : 24,130
  Trades retained               : 90.8%
  Trades with Y label           : 22,266
  Level 1 anomaly rate          : 42.7%
  Level 2 anomaly rate          : 27.0%
